<a href="https://colab.research.google.com/github/Hassanmufezshaikh/RAG-Pipeline-LangChain-and-Gemini-Colab/blob/main/Chat_with_Your_PDFs_(RAG_Application).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Augmented Generation (RAG) Pipeline with LangChain and Gemini

This notebook demonstrates a basic RAG pipeline using LangChain, Google Gemini, and ChromaDB. We will load a PDF document, split it into chunks, create embeddings, store them in a vector database, and then use a language model to answer questions based on the retrieved information.

## 1. Install Necessary Libraries

First, we install all the required Python packages for LangChain, Google Gemini integrations, ChromaDB, and PDF loading. The `-U` flag ensures that the packages are updated to their latest versions.

In [ ]:
!pip install -U \
langchain \
langchain-community \
langchain-core \
langchain-google-genai \
chromadb \
pypdf \
langchain-chroma \
langchain_text_splitters

## 2. Set Up Google API Key

---



To use Google Gemini models, you need an API key. You can get one from Google AI Studio. It's recommended to store your API key securely, for example, using Colab's `userdata` or as an environment variable.

In [ ]:
import os

# Replace "YOUR_KEY" with your actual Google API Key
# For better security, consider using Colab's Secrets feature
os.environ["GOOGLE_API_KEY"] = "place-your-own-gemini-key"

## 3. Load and Split the Document

We'll load a PDF document using `PyPDFLoader`, which converts each page into a `Document` object. Then, we use `RecursiveCharacterTextSplitter` to break down these documents into smaller, manageable `chunks` to ensure they fit within the language model's context window and improve retrieval accuracy.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving AI_Engineer_Roadmap_Step_by_Step.pdf to AI_Engineer_Roadmap_Step_by_Step.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import files

# Upload your PDF file. If you already uploaded 'AI_Engineer_Roadmap_Step_by_Step.pdf',
# you might not need to run this again unless you want to upload a new file.
# uploaded = files.upload()

# Assuming the file 'AI_Engineer_Roadmap_Step_by_Step.pdf' is already in the /content directory
loader = PyPDFLoader("AI_Engineer_Roadmap_Step_by_Step.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 4


## 4. Initialize Embeddings and Vector Database

We use `GoogleGenerativeAIEmbeddings` to convert our document chunks into numerical vector representations. These embeddings capture the semantic meaning of the text. We then store these embeddings in a `Chroma` vector database, which allows for efficient similarity searches.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector DB Created Successfully")

Vector DB Created Successfully


## 5. Initialize the Language Model (LLM)

We initialize the `ChatGoogleGenerativeAI` model, which will be used to generate answers based on the context retrieved from our vector database.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

## 6. Define a Query and Retrieve Relevant Documents

We define a `query` and use the `vector_db` to find the most semantically similar documents (chunks) from our database. These `results` (documents) will serve as context for the language model.

In [ ]:
query = "What is this document about?"

results = vector_db.similarity_search(
    query,
    k=3 # Retrieve top 3 most similar documents
)

print("Retrieved documents:")
for doc in results:
    print(f"- {doc.page_content[:100]}...")

Retrieved documents:
- AI Engineer Roadmap: Step-by-Step Learning
 Guide
This roadmap is designed for a developer who has a...
- Phase 1: RAG + Vector Databases
 Learn embeddings, chunking, semantic search, vector databases, and...
-  Build an automated agent scoring framework.
 Tech Stack: Python, Gemini.
Phase 5: Multi-Agent Cus...


## 7. Construct the Prompt

Now, we construct a prompt for the LLM. This prompt includes the original question (`query`) and the `context` extracted from the retrieved documents. This approach allows the LLM to generate an answer grounded in the provided information.

In [ ]:
context = "\n".join(
    [doc.page_content for doc in results]
)

prompt = f"""
Answer based on context.

Context:
{context}

Question:
{query}
"""

print("Prompt created successfully.")

Prompt created successfully.


## 8. Generate Response

Finally, we invoke the language model (`llm`) with our constructed `prompt` to get an answer.

In [ ]:
response = llm.invoke(prompt)

print("\n--- LLM Response ---")
print(response.content)


--- LLM Response ---
This document is an **AI Engineer Roadmap: Step-by-Step Learning Guide**.

It outlines a structured learning path for developers who have already completed Agentic AI projects and want to become production-ready AI Engineers, detailing various phases, concepts to learn, projects to build, and recommended tech stacks and resources.


In [ ]:
print(vector_db._collection.count())

4


In [ ]:
data = vector_db._collection.get()

print(data.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])


In [ ]:
data = vector_db._collection.get()

print(data['documents'][0])

AI Engineer Roadmap: Step-by-Step Learning
 Guide
This roadmap is designed for a developer who has already completed Agentic AI projects and
wants to become a production-ready AI Engineer. Follow the projects in order and complete each
phase before moving to the next.


In [ ]:
print(data['metadatas'][2])

{'source': 'AI_Engineer_Roadmap_Step_by_Step.pdf', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'subject': '(unspecified)', 'creator': '(unspecified)', 'keywords': '', 'title': '(anonymous)', 'page_label': '2', 'author': '(anonymous)', 'total_pages': 3, 'moddate': '2026-06-05T14:29:57+00:00', 'page': 1, 'trapped': '/False', 'creationdate': '2026-06-05T14:29:57+00:00'}


# Understanding Similarity Search in a Vector Database

In traditional databases such as MySQL and MongoDB, data is searched using exact values, filters, indexes, or keywords.

For example:

### MongoDB Query

```javascript
db.documents.find({
  topic: "RAG"
})
```

This query works only when the exact keyword exists in the database.

Suppose a document contains:

> Retrieval-Augmented Generation combines retrieval systems with large language models.

If a user searches:

> What is RAG?

A traditional database cannot automatically understand that **RAG** and **Retrieval-Augmented Generation** refer to the same concept.

---

# How Vector Databases Solve This Problem

A Vector Database searches based on **meaning** instead of exact keywords.

Rather than storing only text, it stores a numerical representation of the text called an **Embedding**.

### Example

Text:

```text
What is RAG?
```

Embedding:

```text
[0.55, 0.22, 0.91, ...]
```

Similarly, every document chunk is converted into its own vector.

```text
Chunk 1 → [0.12, 0.44, 0.78, ...]

Chunk 2 → [0.81, 0.19, 0.32, ...]

Chunk 3 → [0.63, 0.71, 0.15, ...]
```

These vectors capture the semantic meaning of the text.

---

# What Happens During Similarity Search?

When a user asks:

```text
What is RAG?
```

The embedding model converts the question into a vector.

```text
Question Vector

[0.55, 0.22, 0.91, ...]
```

The Vector Database then compares this vector against all stored chunk vectors.

```text
Question Vector
        │
        ▼

Compare with Chunk 1 Vector

Compare with Chunk 2 Vector

Compare with Chunk 3 Vector

Compare with Chunk 4 Vector

Compare with Chunk 5 Vector

...
```

To measure similarity, the database uses a mathematical technique called **Cosine Similarity**.

The chunks with the highest similarity scores are considered the most relevant to the user's question.

---

# ChromaDB Similarity Search

In our project, this happens using:

```python
results = vector_db.similarity_search(
    query,
    k=3
)
```

### Parameters

- `query` → User's question
- `k=3` → Return the top 3 most relevant chunks

### Output

```text
Chunk 12
Chunk 27
Chunk 31
```

These are the chunks whose meanings are most similar to the user's question.

---

# Mental Model

## Traditional Database (MySQL / MongoDB)

Stores:

- Rows
- Tables
- JSON Documents

Searches Using:

- Keywords
- Filters
- Exact Matches

Example:

```text
Find documents containing the word "RAG"
```

---

## Vector Database (ChromaDB / Pinecone / Weaviate)

Stores:

- Text Chunks
- Embeddings (Vectors)

Searches Using:

- Meaning
- Context
- Semantic Similarity

Example:

```text
Find documents related to
"Retrieval-Augmented Generation"
even if the exact words are not present.
```

---

# Comparison

| Traditional Database | Vector Database |
|----------|----------|
| Stores rows/documents | Stores embeddings |
| Keyword search | Semantic search |
| Exact matching | Meaning matching |
| SQL/Mongo Queries | Similarity Search |
| Structured data | Unstructured data |
| Filters and indexes | Vector similarity |

---

# RAG Retrieval Flow

```text
User Question
        │
        ▼
Convert Question to Embedding
        │
        ▼
Vector Database (ChromaDB)
        │
        ▼
Compare Against All Chunk Vectors
        │
        ▼
Calculate Similarity Scores
        │
        ▼
Return Top K Chunks
        │
        ▼
Provide Context to Gemini
        │
        ▼
Generate Final Answer
```

---

# Key Takeaway

A Traditional Database answers:

> Does this document contain these words?

A Vector Database answers:

> Which document has the most similar meaning to this question?

This ability to search based on meaning rather than keywords is what makes modern RAG (Retrieval-Augmented Generation) systems possible.